In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("data/phishing_email.csv")

In [6]:
df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [7]:
df.columns

Index(['text_combined', 'label'], dtype='object')

In [8]:
df.shape

(82486, 2)

In [9]:
df['label'].value_counts()

label
1    42891
0    39595
Name: count, dtype: int64

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82486 entries, 0 to 82485
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text_combined  82486 non-null  object
 1   label          82486 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.3+ MB


In [11]:
df[df['label']==1]

,text_combined,label
3503,link dwl g 510 802 11 g wireless pci lan adapt...,1
3504,indemand payperview movies sports wed 15 sep 2...,1
3505,want lose 19 weight try adipren hello special ...,1
3506,cialis xanax valium viagra low price prescript...,1
3507,pre order see image click indelible backscatte...,1
...,...,...
82481,info advantageapartmentscom infoadvantageapart...,1
82482,monkeyorg helpdeskmonkeyorg monkeyorg hi josep...,1
82483,help center infohelpcentercoza_infohelpcenterc...,1
82484,metamask infosofamekarcom verify metamask wall...,1


In [12]:
X = df["text_combined"]
y = df["label"]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
X_train.shape, X_test.shape

((65988,), (16498,))

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [16]:
X_train_tfidf.shape

(65988, 10000)

In [17]:
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4764088 stored elements and shape (65988, 10000)>

In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9844829676324403
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7919
           1       0.98      0.99      0.99      8579

    accuracy                           0.98     16498
   macro avg       0.98      0.98      0.98     16498
weighted avg       0.98      0.98      0.98     16498



In [20]:
test_messages = [
    "URGENT! Your bank account has been blocked. Click this link immediately to verify.",
    "Hey, are we still meeting at 5 pm today?",
    "Congratulations! You won ₹50,000. Click here to claim your prize.",
    "Your OTP is 482913. Do not share it with anyone."
]

test_tfidf = tfidf.transform(test_messages)

probabilities = model.predict_proba(test_tfidf)

for message, prob in zip(test_messages, probabilities):
    print(message)
    print("Legitimate:", round(prob[0] * 100, 2), "%")
    print("Phishing:", round(prob[1] * 100, 2), "%")
    print()

URGENT! Your bank account has been blocked. Click this link immediately to verify.
Legitimate: 0.15 %
Phishing: 99.85 %

Hey, are we still meeting at 5 pm today?
Legitimate: 89.89 %
Phishing: 10.11 %

Congratulations! You won ₹50,000. Click here to claim your prize.
Legitimate: 9.99 %
Phishing: 90.01 %

Your OTP is 482913. Do not share it with anyone.
Legitimate: 46.01 %
Phishing: 53.99 %



In [29]:
import re

def detect_indicators(text):
    text = text.lower()
    indicators = []

    # Urgency
    urgency_words = [
        "urgent", "immediately", "act now",
        "within 24 hours", "verify now", "hurry"
    ]

    if any(word in text for word in urgency_words):
        indicators.append("Urgency language")

    # Sensitive information
    sensitive_words = ["otp", "password", "pin", "cvv", "card number"]
    
    if any(word in text for word in sensitive_words):
        indicators.append("Sensitive information mentioned")
    
    # Request for sensitive information
    request_patterns = [
        r"(send|share|provide|give).{0,20}(otp|password|pin|cvv|card)",
        r"(enter|type).{0,20}(otp|password|pin|cvv|card)"
    ]
    
    if any(re.search(pattern, text) for pattern in request_patterns):
        indicators.append("Request for sensitive information")

    # Suspicious requests
    request_words = [
        "send your otp", "share your otp",
        "enter your password", "confirm your pin"
    ]

    if any(word in text for word in request_words):
        indicators.append("Request for sensitive information")

    # Suspicious links
    if re.search(r'https?://|www\.', text):
        indicators.append("Contains a link")

    # Account threats
    threat_words = [
    "account blocked",
    "account has been blocked",
    "account suspended",
    "account has been suspended",
    "account will be closed",
    "legal action"
]

    if any(word in text for word in threat_words):
        indicators.append("Account threat")

    # Prize/scam
    scam_words = [
        "you won", "winner", "lottery",
        "prize", "claim your reward"
    ]

    if any(word in text for word in scam_words):
        indicators.append("Prize/reward scam language")

    return indicators

In [30]:
message = """
URGENT! Your bank account has been blocked.
Click this link immediately to verify your account:
https://hdfc-secure-login.xyz
"""

detect_indicators(message)

['Urgency language', 'Contains a link', 'Account threat']

In [41]:
def analyze_message(text):
    text_tfidf = tfidf.transform([text])
    phishing_probability = model.predict_proba(text_tfidf)[0][1]

    indicators = detect_indicators(text)
    threat_type = detect_threat_type(text)

    # ML is the main signal
    risk_score = phishing_probability * 100

    # Add supporting evidence
    indicator_weights = {
        "Urgency language": 5,
        "Contains a link": 5,
        "Account threat": 10,
        "Sensitive information mentioned": 2,
        "Request for sensitive information": 15,
        "Prize/reward scam language": 10
    }

    for indicator in indicators:
        risk_score += indicator_weights.get(indicator, 0)

    # Protective language slightly reduces risk
    text_lower = text.lower()

    protective_phrases = [
        "do not share",
        "don't share",
        "never share",
        "do not disclose"
    ]

    if any(phrase in text_lower for phrase in protective_phrases):
        risk_score -= 10

    risk_score = max(0, min(risk_score, 100))

    if risk_score >= 70:
        risk_level = "HIGH"
    elif risk_score >= 40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    if risk_level == "LOW" and threat_type == "Unknown":
        threat_type = "Legitimate / No Threat Detected"

    return {
        "risk_score": round(float(risk_score), 2),
        "risk_level": risk_level,
        "threat_type": threat_type,
        "indicators": indicators
    }

In [42]:
test = """
URGENT! Your bank account has been blocked.
Click this link immediately to verify your account:
https://hdfc-secure-login.xyz
"""

analyze_message(test)

{'risk_score': 100.0,
 'risk_level': 'HIGH',
 'threat_type': 'Account Threat',
 'indicators': ['Urgency language', 'Contains a link', 'Account threat']}

In [43]:
tests = [
    "Hey, are we still meeting at 5 pm today?",
    
    "URGENT! Your bank account has been blocked. "
    "Click https://hdfc-secure-login.xyz immediately to verify.",
    
    "Congratulations! You won ₹50,000. "
    "Click here to claim your prize.",
    
    "Your OTP is 482913. Do not share it with anyone.",
    
    "Please send us your OTP to verify your account."
]

for message in tests:
    print("=" * 80)
    print(message)
    print(analyze_message(message))

Hey, are we still meeting at 5 pm today?
{'risk_score': 10.11, 'risk_level': 'LOW', 'threat_type': 'Legitimate / No Threat Detected', 'indicators': []}
URGENT! Your bank account has been blocked. Click https://hdfc-secure-login.xyz immediately to verify.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Account Threat', 'indicators': ['Urgency language', 'Contains a link', 'Account threat']}
Congratulations! You won ₹50,000. Click here to claim your prize.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Prize Scam', 'indicators': ['Prize/reward scam language']}
Your OTP is 482913. Do not share it with anyone.
{'risk_score': 45.99, 'risk_level': 'MEDIUM', 'threat_type': 'Unknown', 'indicators': ['Sensitive information mentioned']}
Please send us your OTP to verify your account.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Credential Theft', 'indicators': ['Sensitive information mentioned', 'Request for sensitive information']}


In [47]:
def detect_threat_type(text):
    text = text.lower()

    if any(word in text for word in ["you won", "winner", "lottery", "prize", "reward"]):
        return "Prize Scam"

    if any(word in text for word in ["send your otp", "share your otp",
                                     "send us your otp", "password", "pin", "cvv"]):
        return "Credential Theft"

    if any(word in text for word in ["account blocked", "account suspended",
                                     "account has been blocked", "legal action"]):
        return "Account Threat"

    if re.search(r'https?://|www\.', text):
        return "Phishing"
    if any(phrase in text for phrase in [
        "do not share",
        "don't share",
        "never share"
    ]) and "otp" in text:
        return "Legitimate / Security Message"
        
    return "Unknown"

In [48]:
analyze_message(
    "URGENT! Your bank account has been blocked. "
    "Click https://hdfc-secure-login.xyz immediately to verify."
)

{'risk_score': 100.0,
 'risk_level': 'HIGH',
 'threat_type': 'Account Threat',
 'indicators': ['Urgency language', 'Contains a link', 'Account threat']}

In [49]:
for message in tests:
    print("=" * 80)
    print(message)
    print(analyze_message(message))

Hey, are we still meeting at 5 pm today?
{'risk_score': 10.11, 'risk_level': 'LOW', 'threat_type': 'Legitimate / No Threat Detected', 'indicators': []}
URGENT! Your bank account has been blocked. Click https://hdfc-secure-login.xyz immediately to verify.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Account Threat', 'indicators': ['Urgency language', 'Contains a link', 'Account threat']}
Congratulations! You won ₹50,000. Click here to claim your prize.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Prize Scam', 'indicators': ['Prize/reward scam language']}
Your OTP is 482913. Do not share it with anyone.
{'risk_score': 45.99, 'risk_level': 'MEDIUM', 'threat_type': 'Legitimate / Security Message', 'indicators': ['Sensitive information mentioned']}
Please send us your OTP to verify your account.
{'risk_score': 100.0, 'risk_level': 'HIGH', 'threat_type': 'Credential Theft', 'indicators': ['Sensitive information mentioned', 'Request for sensitive information']}


In [50]:
from urllib.parse import urlparse
import re

In [73]:
def analyze_url(url):
    indicators = []
    score = 0

    parsed = urlparse(url if "://" in url else "http://" + url)
    domain = parsed.netloc.lower()
    
    # 1. HTTP instead of HTTPS
    if parsed.scheme == "http":
        indicators.append("Uses HTTP instead of HTTPS")
        score += 15

    # 2. IP address instead of domain
    if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", domain):
        indicators.append("Uses an IP address instead of a domain")
        score += 25

    # @ symbol
    if "@" in url:
        indicators.append("Contains @ symbol")
        score += 25
    
    # Very long URL
    if len(url) > 100:
        indicators.append("Unusually long URL")
        score += 25

    # 5. Too many subdomains
    if not re.match(r"^\d{1,3}(\.\d{1,3}){3}$", domain) and domain.count(".") >= 4:
        indicators.append("Excessive subdomains")
        score += 15

    # 6. Suspicious keywords
    suspicious_words = [
        "login", "verify", "verification",
        "secure", "account", "update",
        "confirm", "signin", "password"
    ]

    found_words = [word for word in suspicious_words if word in url.lower()]

    if found_words:
        indicators.append(
            "Contains suspicious keywords: " + ", ".join(found_words)
        )
        score += 0

    score = min(score, 100)

    if score >= 70:
        risk_level = "HIGH"
    elif score >= 40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    threat_type = "Phishing" if score >= 40 else "Legitimate / No Threat Detected"

    return {
        "risk_score": score,
        "risk_level": risk_level,
        "threat_type": threat_type,
        "indicators": indicators
    }

In [74]:
analyze_url("https://google.com")

{'risk_score': 0,
 'risk_level': 'LOW',
 'threat_type': 'Legitimate / No Threat Detected',
 'indicators': []}

In [65]:
analyze_url("http://192.168.1.20/login/verify")

{'risk_score': 45,
 'risk_level': 'MEDIUM',
 'threat_type': 'Phishing',
 'indicators': ['Uses HTTP instead of HTTPS',
  'Uses an IP address instead of a domain',
  'Contains suspicious keywords: login, verify']}

In [66]:
analyze_url("https://accounts.security.example.com/login")

{'risk_score': 5,
 'risk_level': 'LOW',
 'threat_type': 'Legitimate / No Threat Detected',
 'indicators': ['Contains suspicious keywords: login, account']}

In [67]:
analyze_url("https://accounts.security.example.com/login")

{'risk_score': 5,
 'risk_level': 'LOW',
 'threat_type': 'Legitimate / No Threat Detected',
 'indicators': ['Contains suspicious keywords: login, account']}

In [69]:
analyze_url("https://accounts.security.example.com/login")

{'risk_score': 0,
 'risk_level': 'LOW',
 'threat_type': 'Legitimate / No Threat Detected',
 'indicators': ['Contains suspicious keywords: login, account']}

In [70]:
analyze_url("http://192.168.1.20/login/verify")

{'risk_score': 40,
 'risk_level': 'MEDIUM',
 'threat_type': 'Phishing',
 'indicators': ['Uses HTTP instead of HTTPS',
  'Uses an IP address instead of a domain',
  'Contains suspicious keywords: login, verify']}

In [76]:
analyze_url("http://secure-login@evil.com/verify")

{'risk_score': 40,
 'risk_level': 'MEDIUM',
 'threat_type': 'Phishing',
 'indicators': ['Uses HTTP instead of HTTPS',
  'Contains @ symbol',
  'Contains suspicious keywords: login, verify, secure']}

In [72]:
analyze_url("https://very-long-suspicious-login-verification-example.com/" + "a"*100)

{'risk_score': 15,
 'risk_level': 'LOW',
 'threat_type': 'Legitimate / No Threat Detected',
 'indicators': ['Unusually long URL',
  'Contains suspicious keywords: login, verification']}

In [77]:
type(model)
type(tfidf)

sklearn.feature_extraction.text.TfidfVectorizer

In [78]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(tfidf, "vectorizer.pkl")

['vectorizer.pkl']

In [79]:
import os

print(os.path.exists("model.pkl"))
print(os.path.exists("vectorizer.pkl"))

True
True
